# LLM + SAT/SMT Solver

## What This Is
LLMs excel at understanding natural language and generating code, but they make arithmetic and logical errors. SMT solvers are sound and complete for their domain but require formal constraint specifications.

The hybrid approach:
1. **LLM** reads natural language problem, extracts variables and constraints, generates Z3 Python code
2. **Z3** solves the formal constraint system
3. **LLM** interprets the solution and formulates a natural language response

This combines LLM's language understanding with solver's guaranteed correctness.

In [ ]:
import json
from typing import Dict, List, Optional, Tuple

try:
    from z3 import *
    HAS_Z3 = True
except ImportError:
    HAS_Z3 = False

# LLM -> Z3 constraint pipeline
# In production: replace LLMConstraintExtractor with actual API call

class LLMConstraintExtractor:
    """Simulates LLM extraction of constraints from natural language."""
    
    def extract(self, problem: str) -> Dict:
        """Extract variable types and constraints from problem description."""
        # In production: call LLM API to generate this JSON
        # Here we return pre-written extractions for demo problems
        extractions = {
            'scheduling': {
                'variables': [('t1', 'int', 0, 23), ('t2', 'int', 0, 23), ('t3', 'int', 0, 23)],
                'constraints': [
                    {'type': 'min_gap', 'vars': ['t1', 't2'], 'gap': 2},
                    {'type': 'min_gap', 'vars': ['t2', 't3'], 'gap': 1},
                    {'type': 'range', 'var': 't1', 'lo': 9, 'hi': 17},
                    {'type': 'range', 'var': 't2', 'lo': 9, 'hi': 17},
                    {'type': 'range', 'var': 't3', 'lo': 9, 'hi': 17},
                ],
                'objective': 'minimize_sum',
            },
            'resource_allocation': {
                'variables': [('x1', 'real', 0, 100), ('x2', 'real', 0, 100), ('x3', 'real', 0, 100)],
                'constraints': [
                    {'type': 'sum_le', 'vars': ['x1', 'x2', 'x3'], 'bound': 100},
                    {'type': 'ratio', 'var1': 'x1', 'var2': 'x2', 'min_ratio': 0.5},
                ],
                'objective': 'maximize_sum',
            }
        }
        # Return the first matching extraction
        for key, val in extractions.items():
            if key in problem.lower():
                return val
        return extractions['scheduling']  # default

def solve_with_z3(extraction: Dict) -> Optional[Dict]:
    """Execute Z3 solver on extracted constraints."""
    if not HAS_Z3:
        return {'status': 'z3_not_available', 'note': 'Install z3-solver to run'}
    
    vars_z3 = {}
    for name, vtype, lo, hi in extraction['variables']:
        if vtype == 'int':
            vars_z3[name] = Int(name)
        else:
            vars_z3[name] = Real(name)
    
    s = Optimize() if 'objective' in extraction else Solver()
    
    # Add range constraints
    for name, vtype, lo, hi in extraction['variables']:
        s.add(vars_z3[name] >= lo)
        s.add(vars_z3[name] <= hi)
    
    # Add other constraints
    for c in extraction['constraints']:
        if c['type'] == 'min_gap':
            v1, v2 = vars_z3[c['vars'][0]], vars_z3[c['vars'][1]]
            s.add(Or(v2 - v1 >= c['gap'], v1 - v2 >= c['gap']))
        elif c['type'] == 'sum_le':
            total = sum(vars_z3[v] for v in c['vars'])
            s.add(total <= c['bound'])
    
    # Objective
    if 'objective' in extraction and hasattr(s, 'minimize'):
        total = sum(vars_z3[name] for name, _, _, _ in extraction['variables'])
        if extraction['objective'] == 'minimize_sum':
            s.minimize(total)
        else:
            s.maximize(total)
    
    result = s.check()
    if result == sat:
        m = s.model()
        solution = {}
        for name in vars_z3:
            try:
                solution[name] = int(m[vars_z3[name]].as_long())
            except:
                try:
                    solution[name] = float(m[vars_z3[name]].as_decimal(4))
                except:
                    solution[name] = str(m[vars_z3[name]])
        return {'status': 'sat', 'solution': solution}
    else:
        return {'status': 'unsat'}

# Test the pipeline
problems = [
    'Create a scheduling plan with 3 meetings (t1, t2, t3) within work hours',
]

extractor = LLMConstraintExtractor()
print('LLM -> Z3 Constraint Pipeline')
print('=' * 50)
for problem in problems:
    print(f'Problem: {problem}')
    extraction = extractor.extract(problem)
    print(f'Extracted {len(extraction["variables"])} variables, {len(extraction["constraints"])} constraints')
    result = solve_with_z3(extraction)
    print(f'Result: {result}')
    print()
